# RAG over MyFixit repair guides

Small RAG pipeline I built as part of my internship project.

The dataset is MyFixit (a JSON copy of iFixit Mac/phone/console/etc. repair
guides). The pipeline:

1. Load all 15 category JSON files
2. Flatten guides into individual repair steps
3. Build a small text "chunk" per step (title + subject + tools + instruction)
4. Embed with `all-MiniLM-L6-v2`
5. Index with FAISS (cosine similarity via inner product on L2-normed vectors)
6. Retrieve top-k chunks for a query and feed them to a local Mistral via Ollama
7. Evaluate with Precision@K + grounding rate

In [1]:
import os
import re
import glob
import subprocess

import numpy as np
import pandas as pd
import torch
import faiss
import ollama
from sentence_transformers import SentenceTransformer

/Users/asim/Desktop/RAG/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Config

Everything that I might want to tweak goes here so I don't have to hunt for it.

In [2]:
DATASET_DIR  = "./MyFixit-Dataset"
SUBSET_SIZE  = None     # set to e.g. 2000 to test on a slice

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
OLLAMA_MODEL = "mistral"

TOP_K        = 5
MAX_CONTEXT  = 4        # how many chunks actually go into the prompt

INDEX_PATH   = "faiss_index.bin"
CHUNKS_PATH  = "rag_chunks.parquet"

In [3]:
# Pick the fastest device available.
# On my Mac this ends up being "mps".
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

In [4]:
# Quick sanity check that Ollama is actually running and the model is pulled,
# otherwise the generation cells will just blow up later with a confusing error.
try:
    out = subprocess.run(
        ["ollama", "list"],
        capture_output=True, text=True, timeout=5,
    )
    lines = out.stdout.strip().splitlines()[1:]
    models = [line.split()[0] for line in lines if line.strip()]
    has_model = any(OLLAMA_MODEL in m for m in models)

    print(f"Ollama running  |  {len(models)} model(s) available")
    if has_model:
        print(f"  ✓  '{OLLAMA_MODEL}' ready")
    else:
        print(f"  ⚠  '{OLLAMA_MODEL}' not found — run:  ollama pull {OLLAMA_MODEL}")
        print("  ⚠  Generation cells will fail until the model is pulled.")
except Exception:
    print("  ✗  Ollama not running — start it:  ollama serve")
    print("  ✗  Generation cells will fail until Ollama is running.")

print(f"\nDevice  : {DEVICE}")
print(f"Dataset : 15 categories — full MyFixit dataset")
print(f"Embed   : {EMBED_MODEL}")
print(f"LLM     : {OLLAMA_MODEL} via Ollama (local, free, MPS)")

Ollama running  |  1 model(s) available
  ✓  'mistral' ready

Device  : mps
Dataset : 15 categories — full MyFixit dataset
Embed   : sentence-transformers/all-MiniLM-L6-v2
LLM     : mistral via Ollama (local, free, MPS)


## Load the dataset

The repo ships 15 JSON files (one per device category) inside `/jsons`. Most
of them are JSONL (one guide per line) but a couple aren't, so I try both.

In [5]:
def load_full_dataset(dataset_dir):
    repo_url = "https://github.com/rub-ksv/MyFixit-Dataset.git"

    # Clone on first run, otherwise just point at what's already there.
    if not os.path.isdir(dataset_dir):
        print(f"Cloning full dataset into '{dataset_dir}' ...")
        subprocess.run(["git", "clone", repo_url, dataset_dir], check=True)
    else:
        print(f"Dataset found at '{dataset_dir}'")

    json_dir = os.path.join(dataset_dir, "jsons")
    json_files = sorted(glob.glob(os.path.join(json_dir, "*.json")))
    if not json_files:
        raise FileNotFoundError(f"No JSON files found in {json_dir}")

    print(f"Found {len(json_files)} category files:")

    all_dfs = []
    for path in json_files:
        fname = os.path.basename(path)
        try:
            tmp = pd.read_json(path, lines=True)
        except ValueError:
            # not JSONL, try the regular JSON reader
            tmp = pd.read_json(path)

        tmp["source_file"] = fname.replace(".json", "")  # remember which category
        all_dfs.append(tmp)
        print(f"  {fname:<30} {len(tmp):>6,} guides")

    df = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTotal loaded : {len(df):,} guides across {len(all_dfs)} categories")
    return df


df = load_full_dataset(DATASET_DIR)
df.head(2)

Dataset found at './MyFixit-Dataset'
Found 15 category files:
  Apparel.json                      250 guides
  Appliance.json                    609 guides
  Camera.json                     1,717 guides
  Car and Truck.json                286 guides
  Computer Hardware.json            515 guides
  Electronics.json                1,546 guides
  Game Console.json                 705 guides
  Household.json                    940 guides
  Mac.json                        2,224 guides
  Media Player.json                 432 guides
  PC.json                         3,592 guides
  Phone.json                      3,682 guides
  Skills.json                        53 guides
  Tablet.json                     2,272 guides
  Vehicle.json                      172 guides

Total loaded : 18,995 guides across 15 categories


,Title,Ancestors,Guidid,Category,Toolbox,Url,Steps,Subject,source_file
0,Clothing Ribbon Button Replacement,"[Clothing, Apparel, Root]",10262,patagonia clothing,"[{'Name': 'utility scissors', 'Url': 'http://w...",https://www.ifixit.com/Guide/Clothing+Ribbon+B...,"[{'Order': 0, 'Lines': [{'Text': 'examine the ...",Ribbon Button,Apparel
1,How to remove scratches from a Breitling Navit...,"[Watch, Apparel, Root]",13521,Breitling Navitimer,"[{'Name': 'rotary tool', 'Url': 'http://www.if...",https://www.ifixit.com/Guide/How+to+remove+scr...,"[{'Order': 0, 'Lines': [{'Text': ""'''notice th...",,Apparel


In [6]:
def extract_steps(df):
    """Flatten guides into one row per repair step."""
    rows = []
    for _, guide in df.iterrows():
        guide_meta = {
            "guidid": guide.get("Guidid"),
            "title": guide.get("Title"),
            "category": guide.get("Category"),
            "source_file": guide.get("source_file"),
            "subject": guide.get("Subject"),
            "url": guide.get("Url"),
        }

        steps = guide.get("Steps")
        if not isinstance(steps, list):
            continue

        for step in steps:
            row = dict(guide_meta)
            row["step_order"]      = step.get("Order")
            row["step_id"]         = step.get("StepId")
            row["text_raw"]        = step.get("Text_raw")
            row["tools_annotated"] = step.get("Tools_annotated")
            row["tools_extracted"] = step.get("Tools_extracted")
            row["parts_clean"]     = step.get("Word_level_parts_clean")
            row["removal_verbs"]   = step.get("Removal_verbs")
            rows.append(row)

    steps_df = pd.DataFrame(rows)
    cats = sorted(steps_df["source_file"].dropna().unique().tolist())
    print(f"Extracted  : {len(steps_df):,} steps from {df['Guidid'].nunique():,} guides")
    print(f"Categories : {cats}")
    return steps_df


steps_df = extract_steps(df)
steps_df.head(2)

Extracted  : 235,549 steps from 18,995 guides
Categories : ['Apparel', 'Appliance', 'Camera', 'Car and Truck', 'Computer Hardware', 'Electronics', 'Game Console', 'Household', 'Mac', 'Media Player', 'PC', 'Phone', 'Skills', 'Tablet', 'Vehicle']


,guidid,title,category,source_file,subject,url,step_order,step_id,text_raw,tools_annotated,tools_extracted,parts_clean,removal_verbs
0,10262,Clothing Ribbon Button Replacement,patagonia clothing,Apparel,Ribbon Button,https://www.ifixit.com/Guide/Clothing+Ribbon+B...,0,37786,Examine the garment where your ribbon button c...,None,[NA],None,None
1,10262,Clothing Ribbon Button Replacement,patagonia clothing,Apparel,Ribbon Button,https://www.ifixit.com/Guide/Clothing+Ribbon+B...,1,37787,"Tuck one end of the ribbon into the pocket, in...",None,[NA],None,None


## Preprocess the steps

Three small helpers — one for normal text, one for the list-typed columns
(tools / parts), and one for the verbs which arrive as a list of dicts.

In [7]:
def normalize_text(s):
    if s is None:
        return ""
    return re.sub(r"\s+", " ", str(s)).strip()


def clean_list_field(x):
    """Strip whitespace, drop blanks and the literal strings 'NA' / 'NaN'."""
    if not isinstance(x, list):
        return []
    out = []
    for item in x:
        if item is None:
            continue
        s = str(item).strip()
        if not s:
            continue
        if s.upper() in ("NA", "NAN"):
            continue
        out.append(s)
    return out


def extract_verb_names(x):
    # Removal_verbs is a list of {"name": ...} dicts. Pull the names, dedup.
    if not isinstance(x, list):
        return []
    seen = set()
    out = []
    for item in x:
        if not isinstance(item, dict) or "name" not in item:
            continue
        name = str(item["name"]).strip()
        if name and name not in seen:
            seen.add(name)
            out.append(name)
    return out


def preprocess(steps_df):
    df = steps_df.copy()
    df["text_clean"]            = df["text_raw"].apply(normalize_text)
    df["tools_annotated_clean"] = df["tools_annotated"].apply(clean_list_field)
    df["tools_extracted_clean"] = df["tools_extracted"].apply(clean_list_field)
    df["parts_clean_final"]     = df["parts_clean"].apply(clean_list_field)
    df["verbs_clean"]           = df["removal_verbs"].apply(extract_verb_names)

    # Prefer the manually annotated tool list, fall back to the auto-extracted one.
    df["tools_final"] = df.apply(
        lambda r: r["tools_annotated_clean"] if r["tools_annotated_clean"] else r["tools_extracted_clean"],
        axis=1,
    )

    n_empty = (df["text_clean"] == "").sum()
    print(f"Preprocessed : {len(df):,} steps  |  {n_empty:,} empty text (dropped at chunk stage)")
    return df


steps_df = preprocess(steps_df)
steps_df[["text_clean", "tools_final", "parts_clean_final", "verbs_clean"]].head(3)

Preprocessed : 235,549 steps  |  0 empty text (dropped at chunk stage)


,text_clean,tools_final,parts_clean_final,verbs_clean
0,Examine the garment where your ribbon button c...,[],[],[]
1,"Tuck one end of the ribbon into the pocket, in...",[],[],[]
2,"Holding the ribbon in place, slide the garment...",[sewing machine],[],[]


## Build chunks

One chunk = one repair step, formatted as a short labeled blob. I went back
and forth on this — at one point I was just embedding the raw instruction,
but adding the title + subject + tools made retrieval noticeably better
(steps without that context all look the same when you ask "how do I remove
the screws").

In [8]:
def list_to_str(items):
    if isinstance(items, list) and items:
        return ", ".join(str(x) for x in items)
    return "none"


def build_chunks(steps_df):
    df = steps_df.copy()

    df["chunk_text"] = (
        "Title: "          + df["title"].fillna("").astype(str)
        + "\nSubject: "    + df["subject"].fillna("").astype(str)
        + "\nStep: "       + df["step_order"].fillna("").astype(str)
        + "\nTools: "      + df["tools_final"].apply(list_to_str)
        + "\nParts: "      + df["parts_clean_final"].apply(list_to_str)
        + "\nVerbs: "      + df["verbs_clean"].apply(list_to_str)
        + "\nInstruction: " + df["text_clean"].fillna("").astype(str)
    )

    # Drop steps with no actual instruction — they hurt retrieval and add noise.
    rag_df = df[df["text_clean"] != ""].reset_index(drop=True)
    print(f"Chunks created : {len(rag_df):,}")
    print("\nExample chunk:")
    print(rag_df.loc[0, "chunk_text"])
    return rag_df


rag_df = build_chunks(steps_df)

Chunks created : 235,549

Example chunk:
Title: Clothing Ribbon Button Replacement
Subject: Ribbon Button
Step: 0
Tools: none
Parts: none
Verbs: none
Instruction: Examine the garment where your ribbon button came off. There should be a little pocket where the ribbon came out. Stitch rip the seam around the little pocket until it is large enough to easily slide the ribbon in.


In [9]:
def deduplicate(rag_df, subset_size=None):
    before = len(rag_df)
    rag_df = rag_df.drop_duplicates(subset=["chunk_text"]).reset_index(drop=True)
    removed = before - len(rag_df)
    print(f"Dedup : {before:,} -> {len(rag_df):,} unique chunks ({removed:,} removed)")

    if subset_size is not None:
        rag_df = rag_df.iloc[:subset_size].copy()
        print(f"Subset applied : {len(rag_df):,} chunks")
    else:
        print("Full dataset   : no subset applied")
    return rag_df


rag_df = deduplicate(rag_df, SUBSET_SIZE)
print(f"\nFinal : {len(rag_df):,} chunks ready for embedding")

Dedup : 235,549 -> 233,814 unique chunks (1,735 removed)
Full dataset   : no subset applied

Final : 233,814 chunks ready for embedding


## Embed and index

First run takes a few minutes on MPS. After that the index + chunk parquet
are cached on disk and we just reload, which makes iterating on the
retrieval / generation cells much faster.

In [10]:
def load_or_embed(rag_df, model_name, index_path, chunks_path, device):
    # Fast path: index already on disk, just reload.
    if os.path.exists(index_path) and os.path.exists(chunks_path):
        print("Saved artifacts found — loading from disk (skipping embedding).")
        encoder = SentenceTransformer(model_name, device=device)
        index   = faiss.read_index(index_path)
        rag_df  = pd.read_parquet(chunks_path)
        print(f"  index : {index.ntotal:,} vectors")
        print(f"  chunks: {len(rag_df):,} rows")
        return encoder, None, index, rag_df

    # Slow path: encode everything from scratch.
    print(f"Loading encoder on {device} ...")
    encoder = SentenceTransformer(model_name, device=device)

    texts = rag_df["chunk_text"].tolist()
    print(f"Encoding {len(texts):,} chunks — first run takes a few minutes ...")

    batch_size = 64 if device == "mps" else 32
    embeddings = encoder.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True,
        batch_size=batch_size,
        normalize_embeddings=False,   # we'll normalize at the FAISS step
    )
    print(f"Embeddings : {embeddings.shape}  dtype={embeddings.dtype}")
    return encoder, embeddings, None, rag_df


encoder, embeddings, preloaded_index, rag_df = load_or_embed(
    rag_df, EMBED_MODEL, INDEX_PATH, CHUNKS_PATH, DEVICE
)

Saved artifacts found — loading from disk (skipping embedding).
  index : 233,814 vectors
  chunks: 233,814 rows


In [11]:
# Columns we actually want to keep around at query time. Dropping the rest
# keeps the parquet small and reload fast.
SAVE_COLS = [
    "guidid", "title", "category", "source_file",
    "subject", "url", "step_order",
    "text_clean", "tools_final", "parts_clean_final", "chunk_text",
]


def build_and_save_index(embeddings, index_path, chunks_path, rag_df):
    # Copy first — faiss.normalize_L2 mutates in place and I don't want to
    # change the original embeddings array.
    emb = embeddings.astype(np.float32).copy()
    faiss.normalize_L2(emb)

    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    print(f"FAISS index built : {index.ntotal:,} vectors  dim={emb.shape[1]}")

    faiss.write_index(index, index_path)
    rag_df[SAVE_COLS].to_parquet(chunks_path, index=False)
    print(f"Saved -> {index_path}")
    print(f"Saved -> {chunks_path}  ({len(rag_df):,} rows)")
    return index


if preloaded_index is not None:
    index = preloaded_index
    print(f"Using preloaded index ({index.ntotal:,} vectors)")
else:
    index = build_and_save_index(embeddings, INDEX_PATH, CHUNKS_PATH, rag_df)

Using preloaded index (233,814 vectors)


## Retrieval

`IndexFlatIP` + L2-normalized vectors == cosine similarity. Brute force is
fine here, the index is small enough.

In [12]:
def retrieve(query, encoder, index, k=TOP_K):
    vec = encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(vec)
    scores, indices = index.search(vec, k)
    return scores[0], indices[0]


# quick smoke test
_scores, _indices = retrieve("How do I replace the battery?", encoder, index)
print("Retrieval working")
print(f"Top result score : {_scores[0]:.4f}")
print(f"Top result title : {rag_df.iloc[_indices[0]]['title']}")

Retrieval working
Top result score : 0.6235
Top result title : APEX AP-GS918 Battery Replacement


## Generation

Stitch the top-k chunks into a context block and ask Mistral with a strict
prompt. The whole point of the rules in the system prompt is to stop the
model from "helping" by adding generic repair advice that isn't in the
context (the classic RAG failure mode).

In [13]:
def build_context(indices, scores, rag_df, max_docs=MAX_CONTEXT):
    blocks = []
    for rank, (idx, score) in enumerate(zip(indices[:max_docs], scores[:max_docs]), start=1):
        row = rag_df.iloc[idx]

        if isinstance(row["tools_final"], list) and row["tools_final"]:
            tools = ", ".join(row["tools_final"])
        else:
            tools = "not specified"

        block = (
            f"[Source {rank} | similarity {score:.4f}]\n"
            f"Guide    : {row['title']}\n"
            f"Category : {row.get('source_file', 'Unknown')}\n"
            f"Subject  : {row['subject']}\n"
            f"Step     : {row['step_order']}\n"
            f"Tools    : {tools}\n"
            f"Instruction: {row['text_clean']}"
        )
        blocks.append(block)

    return "\n\n---\n\n".join(blocks)

In [14]:
SYSTEM_PROMPT = """You are a precise repair assistant. You answer device repair questions strictly using the provided context.

STRICT RULES — follow every one:
1. Use ONLY information from the context. Do not add steps, tips, or knowledge from outside the context.
2. Format your answer as a numbered list. Each step must reference its source like this: [Guide: <title>, Step <number>]
3. List the required tools at the top under "Tools needed:" before the steps. Only list tools that appear in the context.
4. Never include URLs, links, or web addresses in your answer.
5. Never say things like "reassemble in reverse order" or "consult additional resources" unless that exact phrase appears in the context.
6. If the context genuinely has zero relevant information for the question, respond with exactly: "No relevant repair steps were found for this query. Please try a more specific question."
7. Do not contradict yourself. If you say context is insufficient, do not then provide steps.
8. Do not pad the answer. If there is only one relevant step in the context, give one step.

ANSWER FORMAT:
Tools needed: <list tools from context, or "Not specified">
Steps:
1. [Guide: <title>, Step <n>] <instruction from context>
2. [Guide: <title>, Step <n>] <instruction from context>
..."""


def generate(query, context, model=OLLAMA_MODEL):
    user_msg = f"Repair context:\n{context}\n\nQuestion: {query}"
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        options={
            "temperature":    0.0,    # deterministic
            "num_predict":    512,
            "repeat_penalty": 1.15,
            "top_p":          0.9,
        },
    )
    return response["message"]["content"].strip()


# Try one end-to-end.
query   = "How do I remove the hard drive?"
scores, idxs = retrieve(query, encoder, index)
context = build_context(idxs, scores, rag_df)
answer  = generate(query, context)

print(f"QUERY : {query}")
print(answer)

QUERY : How do I remove the hard drive?
Tools needed: Not specified
Steps:
1. [Guide: Replacing the Hard Drive, Step 12] Remove the final two screws (shown) and slide out the hard drive.
2. [Guide: HP ENVY Rove 20-k014us Hard Drive Removal, Step 10] Gently remove the hard drive from the computer. Unplug the hard drive.
3. [Guide: Asus VivoBook X540SA-BPD0602V Hard Drive Removal Disassembly, Step 12] The Hard Drive is located in the bottom right hand corner. It should be the device covered in black foam. Do not remove the foam.
4. [Guide: Disassembling HP Compaq dx2000MT Hard Drive, Step 11] Remove the hard drive from the hard drive cage by pulling it outward from the cage


## Convenience wrapper

In [15]:
def ask(question, show_sources=True):
    print(f"Q: {question}")

    scores, idxs = retrieve(question, encoder, index, k=TOP_K)
    context      = build_context(idxs, scores, rag_df, max_docs=MAX_CONTEXT)
    answer       = generate(question, context)

    print(answer)

    if show_sources:
        print("\nSources:")
        for rank, (idx, score) in enumerate(zip(idxs[:MAX_CONTEXT], scores[:MAX_CONTEXT]), start=1):
            row = rag_df.iloc[idx]
            print(f"  {rank}. [{score:.4f}]  {row['title']}  |  Step {row['step_order']}")


ask("How do I replace the battery?")
ask("How do I remove the keyboard?")
ask("What tools do I need to open the MacBook case?")

Q: How do I replace the battery?
Tools needed: Not specified
Steps:
1. [Source 1, Step 1] Squeeze the plastic opening tool in between the seam where the screen joins the plastic case. Begin separating the bottom of the case from the screen by carefully going around the whole edge of the case and slowly prying it apart. Be careful when separating the lids because the wires holding the speaker to the motherboard are very short.
2. [Source 4, Step 2] Once the cover is removed, locate the motherboard on the back shell (the left component in the third image). The motherboard will be the middle component as shown in the third image. To extract the motherboard, remove the seven Phillips head screws used to fasten the motherboard to the unit.
3. [Source 4, Not specified] Be aware of the power button that is located between the back shell and the motherboard on the lower right corner.
4. [Source 4, Not specified] The battery is located on the motherboard. Two wires, red and black, are soldered 

## Evaluation

Two metrics:

- **Precision@K** — of the top-K retrieved chunks, how many contain at
  least one expected keyword. Crude but it tells me whether retrieval is
  pulling relevant stuff at all.
- **Grounding rate** — does the *answer* mention the expected keywords,
  and is it not just a refusal. This catches cases where retrieval works
  but the model bails out anyway.

I also track a small set of canned phrases the model has been caught
hallucinating in earlier runs (e.g. "reassemble in reverse order" — that
one is suspiciously common when the model runs out of real steps).

In [16]:
assert all(v in globals() for v in ["df", "steps_df", "rag_df", "encoder", "index"]), \
    "Run the earlier cells in order first"


test_cases = [
    # Computers / Mac / PC
    {"query": "How do I remove the hard drive?",
     "keywords": ["hard drive", "screw", "remove", "cable"]},
    {"query": "How do I replace the battery?",
     "keywords": ["battery", "screw", "remove", "replace"]},
    {"query": "How do I remove the keyboard?",
     "keywords": ["keyboard", "screw", "ribbon", "remove"]},
    # Phone / tablet
    {"query": "How do I replace an iPhone screen?",
     "keywords": ["screen", "display", "remove", "screw", "cable"]},
    {"query": "How do I replace a tablet battery?",
     "keywords": ["battery", "remove", "back", "replace"]},
    # Console
    {"query": "How do I open a PlayStation 4?",
     "keywords": ["screw", "cover", "remove", "open"]},
    # Camera
    {"query": "How do I fix a camera lens?",
     "keywords": ["lens", "screw", "remove", "clean"]},
]

# Phrases the model has been caught hallucinating before.
HALLUCINATION_PHRASES = [
    "reassemble in reverse",
    "consult additional",
    "align the positive and negative",
    "insert the new one in its place",
]


K = 3
retrieval_precision = []
grounded = 0
hallucinated = 0

print("EVALUATION — 15-category RAG pipeline")

for tc in test_cases:
    q   = tc["query"]
    kws = tc["keywords"]

    scores, idxs = retrieve(q, encoder, index, k=K)
    context      = build_context(idxs, scores, rag_df, max_docs=MAX_CONTEXT)
    answer       = generate(q, context)

    # --- Precision@K ---
    hits = 0
    for idx in idxs:
        chunk_text = rag_df.iloc[idx]["text_clean"].lower()
        if any(kw.lower() in chunk_text for kw in kws):
            hits += 1
    prec = hits / K
    retrieval_precision.append(prec)

    # --- Grounding ---
    is_refusal = "no relevant repair steps" in answer.lower()
    if is_refusal:
        matched = []
    else:
        matched = [kw for kw in kws if kw.lower() in answer.lower()]
    is_grounded = bool(matched) and not is_refusal
    if is_grounded:
        grounded += 1

    # --- Hallucination ---
    is_hallucinated = any(p in answer.lower() for p in HALLUCINATION_PHRASES)
    if is_hallucinated:
        hallucinated += 1

    print(f"\nQ : {q}")
    print(f"   Precision@{K}    : {prec:.2f}  ({hits}/{K} relevant hits)")
    print(f"   Matched keywords : {matched}")
    print(f"   Grounded         : {'yes' if is_grounded else 'no'}")
    print(f"   Hallucination    : {'YES' if is_hallucinated else 'clean'}")


n = len(test_cases)
print()
print(f"Dataset       : 15 categories — {df['Guidid'].nunique():,} guides")
print(f"Total steps   : {len(steps_df):,}")
print(f"Indexed chunks: {len(rag_df):,}")
print(f"Embed model   : {EMBED_MODEL}")
print(f"LLM           : {OLLAMA_MODEL} (local, MPS)")
print(f"Avg Precision@{K}: {sum(retrieval_precision)/n:.2f}")
print(f"Grounding rate : {grounded}/{n}  ({grounded/n:.0%})")
print(f"Hallucinations : {hallucinated}/{n}  ({'clean' if hallucinated == 0 else 'review needed'})")

EVALUATION — 15-category RAG pipeline

Q : How do I remove the hard drive?
   Precision@3    : 1.00  (3/3 relevant hits)
   Matched keywords : ['hard drive', 'screw', 'remove']
   Grounded         : yes
   Hallucination    : clean

Q : How do I replace the battery?
   Precision@3    : 0.67  (2/3 relevant hits)
   Matched keywords : ['battery', 'screw', 'remove', 'replace']
   Grounded         : yes
   Hallucination    : clean

Q : How do I remove the keyboard?
   Precision@3    : 1.00  (3/3 relevant hits)
   Matched keywords : ['keyboard', 'ribbon']
   Grounded         : yes
   Hallucination    : clean

Q : How do I replace an iPhone screen?
   Precision@3    : 1.00  (3/3 relevant hits)
   Matched keywords : ['screen', 'display', 'remove', 'cable']
   Grounded         : yes
   Hallucination    : clean

Q : How do I replace a tablet battery?
   Precision@3    : 1.00  (3/3 relevant hits)
   Matched keywords : ['battery', 'remove', 'back', 'replace']
   Grounded         : yes
   Hallucina